# Create coarse cell type annotations

## Automatic coarse aggregation for Supp. Fig. 2

In [1]:
from pathlib import Path
from pprint import pprint

import anndata as ad

from tqdm import tqdm

In [2]:
import obonet
import networkx


url = 'https://github.com/obophenotype/cell-ontology/releases/download/v2025-07-30/cl-simple.obo'
graph = obonet.read_obo(url, ignore_obsolete=True)

# only use "is_a" edges
edges_to_delete = []
for i, x in enumerate(graph.edges):
    if x[2] != 'is_a':
        edges_to_delete.append((x[0], x[1]))
for x in edges_to_delete:
    graph.remove_edge(u=x[0], v=x[1])

# define mapping from id to name
id_to_name = {id_: data.get('name') for id_, data in graph.nodes(data=True)}
# define inverse mapping from name to id
name_to_id = {v: k for k, v in id_to_name.items()}


def find_child_nodes(cell_type):
    return [id_to_name[node] for node in networkx.ancestors(graph, name_to_id[cell_type])]


def find_parent_nodes(cell_type):
    return [id_to_name[node] for node in networkx.descendants(graph, name_to_id[cell_type])]

In [3]:
coarse_ontology_labels = [
    ("T cell", "CL:0000084"),
    ("B cell", "CL:0000236"),
    ("natural killer cell", "CL:0000623"),
    ("monocyte", "CL:0000576"),
    ("dendritic cell", "CL:0000451"),
    ("precursor cell", "CL:0011115"),
    ("blood cell", "CL:0000081"),
]

In [4]:
import numpy as np


def aggregate_to_coarse_cl_terms(
    ct_labels: list,
    coarse_labels: list[tuple[str, str]]
):
    ct_labels = np.unique(ct_labels).tolist()
    coarse_cl_terms = {}
    for ct in ct_labels:
        try:
            parent_nodes = find_parent_nodes(ct) + [ct]
            matched_coarse_terms = [
                coarse_ct[0] for coarse_ct in coarse_labels 
                if coarse_ct[0] in parent_nodes
            ]
            if matched_coarse_terms:
                coarse_cl_terms[ct] = matched_coarse_terms[0]
            else:
                coarse_cl_terms[ct] = None
        except KeyError:
            coarse_cl_terms[ct] = None

    return coarse_cl_terms


In [5]:
DATA_PATH = Path("/vol/data/dataset-similarity/preprocessed")

DATASETS = [
    "7d7cabfd-1d1f-40af-96b7-26a0825a306d",
    "03f821b4-87be-4ff4-b65a-b5fc00061da7_PBMC",
    "4f889ffc-d4bc-4748-905b-8eb9db47a2ed",
    "b0cf0afa-ec40-4d65-b570-ed4ceacc6813",
    "b9fc3d70-5a72-4479-a046-c2cc1ab19efc",
    "ced320a1-29f3-47c1-a735-513c7084d508_CAP",
    "ddfad306-714d-4cc0-9985-d9072820c530",
    "eb735cc9-d0a7-48fa-b255-db726bf365af",
    "ed9185e3-5b82-40c7-9824-b2141590c7f0",
]

In [6]:
for dataset in tqdm(DATASETS):
    adata = ad.read_h5ad(DATA_PATH / f"{dataset}.h5ad")
    coarse_mapping = aggregate_to_coarse_cl_terms(
        adata.obs.cell_type,
        coarse_ontology_labels
    )

    adata.obs["ct_coarse"] = adata.obs["cell_type"].astype(str).replace(coarse_mapping).astype("category")
    adata = adata[adata.obs.ct_coarse.notna()]
    adata.write_h5ad(DATA_PATH / f"{dataset}_coarse_annot.h5ad", compression="gzip")


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [32:33<00:00, 217.00s/it]


## Manual aggregation for Figure 2

In [3]:
import anndata as ad

In [4]:
aida = ad.read_h5ad(
    "/vol/data/dataset-similarity/preprocessed/ced320a1-29f3-47c1-a735-513c7084d508_CAP.h5ad"
)

In [33]:
aida_coarse_mapping = {
    "T Cell": [
        "T_unknown",
        "CD8+_T_GZMBhi",
        "CD8+_T_GZMKhi",
        "CD8+_T_unknown",
        "CD4+_T_cyt",
        "gdT_GZMBhi",
        "gdT_GZMKhi",
        "CD4+_T",
        "CD8+_T_naive",
        "MAIT",
        "CD4+_T_naive",
        "CD4+_T_cm",
        "CD4+_T_em",
        "Treg",
        "dnT",
    ],
    "B Cell": [
        "B_unknown",
        "naive_B",
        "memory_B_IGHMhi",
        "memory_B_IGHMlo",
        "memory_B_IGHMlo",
        "memory_B",
        "atypical_B",
    ],
    "Plasma Cell": [
        "Plasma_Cell"
    ],
    "NK Cell": [
        "NK",
        "CD16+_NK",
        "CD56+_NK",
        "ILC",
    ],
    "Monocyte": [
        "Myeloid",
        "CD14+_Monocyte",
        "CD16+_Monocyte",
    ],
    "Dendritic Cell": [
        "cDC2",
        "cDC1",
        "DC_SIGLEC6hi",
        "pDC",
        
    ],
    "HSPCs": [
        "CD34_HSPC",
    ],
    "Platelets": [
        "Platelet"
    ],
}

mapping_aida = {}
for k, v in aida_coarse_mapping.items():
    for elem in v:
        mapping_aida[elem] = k


In [43]:
aida.obs["ct_coarse"] = aida.obs["author_cell_type"].replace(mapping_aida)

/tmp/ipykernel_247434/217200213.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  aida.obs["ct_coarse"] = aida.obs["author_cell_type"].replace(mapping_aida)


In [44]:
aida.write_h5ad("/vol/data/dataset-similarity/preprocessed/ced320a1-29f3-47c1-a735-513c7084d508_CAP.h5ad")

In [35]:
wijst = ad.read_h5ad(
    "/vol/data/dataset-similarity/preprocessed/7d7cabfd-1d1f-40af-96b7-26a0825a306d.h5ad"
)

In [38]:
wijst_coarse_mapping = {
    "T Cell": [
        "T4_Mem",
        "Tgd_2",
        "T8_Mem",
        "T4_Naive",
        "T8_MAIT",
        "Tgd_1",
        "T4_Treg",
        "T8_Naive",
        "T4_Mem_Prolif",
        "T8_Mem_Prolif",
    ],
    "B Cell": [
        "B_Naive",
        "B_Mem_Prolif",
        "B_Mem",
        "B_Preplasma",
        
    ],
    "Plasma Cell": [
        "PB",
        "PB_Prolif",
        
    ],
    "NK Cell": [
        "NK_CD16+",
        "NKT",
        "NK_CD56++",
        "T_NK_Prolif",
        "NK_Prolif",
        "",
    ],
    "Monocyte": [
        "cM",
        "ncM",
    ],
    "Dendritic Cell": [
        "cDC_2",
        "pDC",
        "cDC_1",
    ],
    "HSPCs": [
        "Progen_CMP",
        "Progen_MPP",
        "Progen_MEP",
        "Progen_CLP",
    ],
    "Platelets": [],
}

mapping_wijst = {}
for k, v in wijst_coarse_mapping.items():
    for elem in v:
        mapping_wijst[elem] = k

In [41]:
wijst.obs["ct_coarse"] = wijst.obs["cell_type_author"].replace(mapping_wijst)

/tmp/ipykernel_247434/1793506385.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  wijst.obs["ct_coarse"] = wijst.obs["cell_type_author"].replace(mapping_wijst)


In [42]:
wijst.write_h5ad("/vol/data/dataset-similarity/preprocessed/7d7cabfd-1d1f-40af-96b7-26a0825a306d.h5ad")